# EJERCICIO SQL: CERVEZAS

In [15]:
import pandas as pd
import sqlite3

### 1. Crear Base De Datos:

In [16]:
# Conectar a la base de datos
connection = sqlite3.connect('cervezas.db')
cursor = connection.cursor()

In [17]:
# Crear tabla cervezas:
cursor.execute('''
    CREATE TABLE IF NOT EXISTS CERVEZAS (
        CodC VARCHAR(2) PRIMARY KEY,
        Envase VARCHAR(10),
        Capacidad REAL,
        Stock INT
    )
''')

In [18]:
# Crear tabla Bares:
cursor.execute('''
    CREATE TABLE IF NOT EXISTS BARES (
        CodB VARCHAR(3) PRIMARY KEY,
        Cif VARCHAR(9),
        Nombre VARCHAR(50),
        Localidad VARCHAR(50)
    )
''')

In [19]:
# Crear tabla Empleados:
cursor.execute('''
    CREATE TABLE IF NOT EXISTS EMPLEADOS (
        CodE VARCHAR(1) PRIMARY KEY,
        Nombre VARCHAR(50),
        Sueldo REAL
        )
''')

In [21]:
# Crear tabla Reparto:
cursor.execute('''
    CREATE TABLE IF NOT EXISTS REPARTO (
        CodE VARCHAR(3),
        CodB VARCHAR(3),
        CodC VARCHAR(3),
        Fecha DATE,
        Cantidad INT,
        PRIMARY KEY (CodE, CodB, CodC),
        FOREIGN KEY (CodE) REFERENCES EMPLEADOS(CodE),
        FOREIGN KEY (CodB) REFERENCES BARES(CodB),
        FOREIGN KEY (CodC) REFERENCES CERVEZAS(CodC)
    )
''')

In [22]:
# Insertar datos en la tabla cervezas:
cursor.executemany("INSERT INTO CERVEZAS VALUES (?,?,?,?)", [
    ('01', 'Botella', 0.2,  3600),
    ('02', 'Botella', 0.33, 1200),
    ('03', 'Lata',    0.33, 2400),
    ('04', 'Botella', 1,    288),
    ('05', 'Barril',  60,   30)
])
connection.commit()

In [23]:
# Insertar datos en la tabla Bares:
cursor.executemany("INSERT INTO BARES VALUES (?,?,?,?)", [
    ('001', '11111111X', 'Stop', 'Villa Botijo'),
    ('002', '22222222Y', 'Las Vegas', 'Villa Botijo'),
    ('003',None,'Club Social','Las Ranas'),
    ('004', '33333333Z','Otra Ronda','La Esponja')
])
connection.commit()

In [24]:
# Insertar datos en la tabla Empleados:
cursor.executemany("INSERT INTO EMPLEADOS VALUES (?,?,?)", [
    ('1', 'Prudencio Caminero', 120000),
    ('2', 'Vicente Merario', 110000),
    ('3', 'Valentin Siempre', 100000)
])
connection.commit()

In [25]:
# Insertar datos en la tabla Reparto:
cursor.executemany("INSERT INTO REPARTO VALUES (?,?,?,?,?)", [
    ('1', '001', '01', '21/10/05', 240),
    ('1', '001', '02', '21/10/05', 48),
    ('1', '002', '03', '22/10/05', 60),
    ('1', '004', '05', '22/10/05', 4),
    ('2', '002', '03', '22/10/05', 48),
    ('2', '002', '05', '23/10/05', 2),
    ('2', '004', '01', '23/10/05', 480),
    ('2', '004', '02', '24/10/05', 72),
    ('3', '003', '03', '24/10/05', 48),
    ('3', '003', '04', '25/10/05', 20)
])
connection.commit()
# He cambiado el formato de la fecha para poder hacer las consultas correctamente, ya que el formato anterior no me lo reconocía como fecha.

In [26]:
# Comprobar que las tablas se han creado correctamente
res = cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
for name in res:
    print(name[0])

CERVEZAS
BARES
EMPLEADOS
REPARTO


In [27]:
# Con esta función leemos los datos y lo pasamos a un DataFrame de Pandas
def sql_query(query):

    # Ejecuta la query
    cursor.execute(query)

    # Almacena los datos de la query 
    ans = cursor.fetchall()

    # Obtenemos los nombres de las columnas de la tabla
    names = [description[0] for description in cursor.description]

    return pd.DataFrame(ans,columns=names)

In [28]:
# Lo mismo de arriba pero con la función para ver que funciona correctamente:
sql_query("SELECT name FROM sqlite_master WHERE type='table'")

,name
0,CERVEZAS
1,BARES
2,EMPLEADOS
3,REPARTO


### 2. Enunciados

In [29]:
# 1. Obtener el nombre de los empleados que hayan repartido al bar **Stop** durante la semana del 17 al 23 de octubre de 2005.
query = '''
SELECT DISTINCT Empleados.Nombre FROM Empleados
INNER JOIN Reparto ON Empleados.CodE = Reparto.CodE
INNER JOIN Bares ON Reparto.CodB = Bares.CodB
WHERE Bares.Nombre = 'Stop' AND Reparto.Fecha BETWEEN '17/10/05' AND '23/10/05'
'''

sql_query(query)

,Nombre
0,Prudencio Caminero


In [30]:
# 2. Obtener el Cif y nombre de los bares a los que se ha repartido cerveza de tipo **Botella** y capacidad inferior a 1 litro, ordenados por localidad.
query = '''
SELECT DISTINCT Bares.Cif, Bares.Nombre FROM Bares
INNER JOIN Reparto ON Bares.CodB = Reparto.CodB
INNER JOIN Cervezas ON Reparto.CodC = Cervezas.CodC
WHERE Envase = 'Botella' AND Capacidad < 1
ORDER BY Localidad
'''
sql_query(query)

,Cif,Nombre
0,33333333Z,Otra Ronda
1,11111111X,Stop


In [31]:
# 3. Obtener los repartos (nombre del bar, envase y capacidad de la bebida, fecha y cantidad) realizados por **Prudencio Caminero**.
query = '''
SELECT Bares.Nombre, Cervezas.Envase, Cervezas.Capacidad, Reparto.Fecha, Reparto.Cantidad FROM Reparto
INNER JOIN Empleados ON Reparto.CodE = Empleados.CodE
INNER JOIN Bares ON Reparto.CodB = Bares.CodB
INNER JOIN Cervezas ON Reparto.CodC = Cervezas.CodC
WHERE Empleados.Nombre = 'Prudencio Caminero'
'''
sql_query(query)

,Nombre,Envase,Capacidad,Fecha,Cantidad
0,Stop,Botella,0.20,21/10/05,240
1,Stop,Botella,0.33,21/10/05,48
2,Las Vegas,Lata,0.33,22/10/05,60
3,Otra Ronda,Barril,60.00,22/10/05,4


In [32]:
# 4. Obtener los bares a los que se les ha repartido envases de tipo **botella** y capacidad 0.2 ó 0.33.
query = '''
SELECT DISTINCT Bares.Nombre FROM Bares
INNER JOIN Reparto ON Bares.CodB = Reparto.CodB
INNER JOIN Cervezas ON Reparto.CodC = Cervezas.CodC
WHERE Cervezas.Envase = 'Botella' AND Cervezas.Capacidad IN (0.2, 0.33)
'''
sql_query(query)

,Nombre
0,Stop
1,Otra Ronda


In [ ]:
# 5. Nombre de los empleados que han repartido a los bares **"Stop"** y **"Las Vegas"** cervezas con envase botella.
query = '''
SELECT DISTINCT Empleados.Nombre FROM Empleados
INNER JOIN Reparto ON Empleados.CodE = Reparto.CodE
INNER JOIN Bares ON Reparto.CodB = Bares.CodB
INNER JOIN Cervezas ON Reparto.CodC = Cervezas.CodC
WHERE Bares.Nombre IN ('Stop', 'Las Vegas') AND Cervezas.Envase = 'Botella'
'''
sql_query(query)

,Nombre
0,Prudencio Caminero


In [ ]:
# Así si esta bien, pero no es correcto porque nos devuelve los empleados que han repartido a Stop o a Las Vegas, 
# y lo que queremos es el nombre de los empleados que han repartido a ambos bares, por lo que tenemos que usar INTERSECT 
# para obtener solo los empleados que han repartido a ambos bares.
query = '''
SELECT Empleados.Nombre FROM Empleados
INNER JOIN Reparto ON Empleados.CodE = Reparto.CodE
INNER JOIN Bares ON Reparto.CodB = Bares.CodB
INNER JOIN Cervezas ON Reparto.CodC = Cervezas.CodC
WHERE Bares.Nombre = 'Stop' AND Cervezas.Envase = 'Botella'
INTERSECT
SELECT Empleados.Nombre FROM Empleados
INNER JOIN Reparto ON Empleados.CodE = Reparto.CodE
INNER JOIN Bares ON Reparto.CodB = Bares.CodB
INNER JOIN Cervezas ON Reparto.CodC = Cervezas.CodC
WHERE Bares.Nombre = 'Las Vegas' AND Cervezas.Envase = 'Botella'
'''
sql_query(query)

,Nombre


In [ ]:
# 6. Obtener el nombre y número de viajes que ha realizado cada empleado fuera de **Villa Botijo**.
query = '''
SELECT Empleados.Nombre, COUNT(Reparto.CodE) AS NumeroDeViajes FROM Empleados
INNER JOIN Reparto ON Empleados.CodE = Reparto.CodE
INNER JOIN Bares ON Reparto.CodB = Bares.CodB
WHERE Bares.Localidad != 'Villa Botijo'
GROUP BY Empleados.Nombre
'''
sql_query(query)

,Nombre,NumeroDeViajes
0,Prudencio Caminero,1
1,Valentin Siempre,2
2,Vicente Merario,2


In [ ]:
# 7. Obtener el nombre y localidad del bar que más litros de cerveza ha comprado.
query = '''
SELECT Bares.Nombre, Bares.Localidad, SUM(Cervezas.Capacidad * Reparto.Cantidad) AS TotalLitros FROM Bares
INNER JOIN Reparto ON Bares.CodB = Reparto.CodB
INNER JOIN Cervezas ON Reparto.CodC = Cervezas.CodC
GROUP BY Bares.Nombre, Bares.Localidad
ORDER BY TotalLitros DESC
LIMIT 1
'''
sql_query(query)
# He añadido también la columna de TotalLitros para ver cuantos litros ha comprado el bar que más ha comprado.

,Nombre,Localidad,TotalLitros
0,Otra Ronda,La Esponja,359.76


In [ ]:
# 8. Obtener los bares que han adquirido todos los tipos de cerveza con envase de botella y capacidad menor que 1 litro.
query = '''
SELECT Bares.Nombre FROM Bares
INNER JOIN Reparto ON Bares.CodB = Reparto.CodB
INNER JOIN Cervezas ON Reparto.CodC = Cervezas.CodC
WHERE Cervezas.Envase = 'Botella' AND Cervezas.Capacidad < 1
GROUP BY Bares.CodB, Bares.Nombre
HAVING COUNT(DISTINCT Cervezas.CodC) = (
    SELECT COUNT(*) 
    FROM Cervezas 
    WHERE Envase = 'Botella' AND Capacidad < 1
)
'''
sql_query(query)
# Having filtra grupos de registros que cumplen con condiciones específicas, en este caso, el numero de tipos de cerveza con envase de botella y capacidad menor de 1 litro que ha comprado cada bar,
# y lo compara con el numero total de tipos de cerveza con envase de botella y capacidad menor de 1 litro que hay en la tabla cervezas. 
# De esta forma, solo se seleccionan los bares que han comprado todos los tipos de cerveza con envase de botella y capacidad menor de 1 litro.

,Nombre
0,Stop
1,Otra Ronda


In [148]:
# 9. Subir un 5% el sueldo del empleado que más días haya trabajado.
query = '''
UPDATE Empleados
SET Sueldo = Sueldo * 1.05
WHERE CodE = (
    SELECT CodE
    FROM Reparto
    GROUP BY CodE
    ORDER BY COUNT(DISTINCT Fecha) DESC
    LIMIT 1
)
'''
cursor.execute(query)
connection.commit()
# En esta consulta, primero se selecciona el CodE del empleado que más días ha trabajado utilizando una subconsulta que cuenta los días distintos en los que cada empleado ha realizado un reparto y ordena,
# los resultados de mayor a menor, limitando la selección al primer resultado (el empleado con más días trabajados). Luego, se actualiza el sueldo de ese empleado incrementándolo en un 5%.

In [151]:
# Comprobar que el sueldo se ha actualizado correctamente:
sql_query("SELECT * FROM Empleados")

,CodE,Nombre,Sueldo
0,1,Prudencio Caminero,120000.00
1,2,Vicente Merario,127338.75
2,3,Valentin Siempre,100000.00


In [ ]:
# 10. Insertar un nuevo reparto del empleado **"Vicente Merario"** al bar **"Stop"** de 48 cervezas de tipo lata el día 10/26/05.
query = '''
INSERT INTO Reparto (CodE, CodB, CodC, Fecha, Cantidad) 
VALUES (
    (SELECT CodE FROM Empleados WHERE Nombre = 'Vicente Merario'),
    (SELECT CodB FROM Bares WHERE Nombre = 'Stop'),
    (SELECT CodC FROM Cervezas WHERE Envase = 'Lata'),
    '26/10/05',
    48)
'''
cursor.execute(query)
connection.commit()
# Para insertar el nuevo reparto, se utilizan subconsultas para obtener los códigos correspondientes al empleado "Vicente Merario", al bar "Stop" y a la cerveza con envase "Lata". 
# Luego, se inserta el nuevo registro en la tabla Reparto con la fecha y cantidad especificadas.
# También se podría haber hecho con los códigos directamente.

In [155]:
# Comprobar que el reparto se ha insertado correctamente:
sql_query("SELECT * FROM Reparto")

,CodE,CodB,CodC,Fecha,Cantidad
0,1,001,01,21/10/05,240
1,1,001,02,21/10/05,48
2,1,002,03,22/10/05,60
3,1,004,05,22/10/05,4
4,2,002,03,22/10/05,48
5,2,002,05,23/10/05,2
6,2,004,01,23/10/05,480
7,2,004,02,24/10/05,72
8,3,003,03,24/10/05,48
9,3,003,04,25/10/05,20
